# 03 — Uncertainty Modeling

This notebook introduces uncertainty analysis using the **EMA (Exploratory Modeling and Analysis) Workbench** to evaluate how deep uncertainties propagate into corridor performance across two snapshot horizons (**Year 1** and **Year 40**).

## Learning Objectives

By the end of this notebook, you will be able to:
1. Register structural deep uncertainties (`u_demand`, `u_beta_pt`) and economic parameters (`PERTURBABLE_PARAMS`) in the **EMA Workbench**.
2. Characterize how uncertainty widens over time from Year 1 to Year 40.
3. Sample the uncertainty space using **Monte Carlo**, **Latin Hypercube (LHS)**, and **Sobol** designs.
4. Quantify physical outcome distributions (`pt_share`, `avg_travel_time_min`, `congestion_delay_hours`).
5. Rank influential uncertainties using **Feature Scoring**, **Morris Screening**, and **Sobol Sensitivity Analysis**.
6. Compare three static pathways under identical uncertainty futures:
   * **`baseline` (Stage 0):** No new infrastructure for 40 years.
   * **`static1` (Stage 1):** Local Stations & Access Package built in Year 1.
   * **`static2` (Stage 2):** Core Tunnel & Full Service Package built in Year 1.
7. Perform **Scenario Discovery (PRIM)** to identify failure boundaries where congestion or emissions exceed thresholds.


## Setup

Run this notebook from either the repository root or the `notebooks` folder. This is the first notebook that needs the **EMA Workbench**, plus `SALib` (which EMA Workbench uses internally for Morris and Sobol sampling/analysis) and `scikit-learn` (used internally for feature scoring).


In [ ]:
# Install the shared project requirements into the active notebook kernel
from pathlib import Path
_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
%pip install --quiet -r "$_PROJECT_ROOT/requirements.txt"


In [ ]:

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

# 1. Project directory setup
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CODE_DIR = PROJECT_ROOT / "code"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CODE_DIR))

# 2. Project modules
import parameters as p
import stages as stages_module
import pathways as pathways_module   # Replaced policies with pathways
import simulation_engine as m
import transport_model_interface as tmi
from parameters import N_YEARS, FIXED_PARAMS, PERTURBABLE_PARAMS
# Ensure ultra-fast LUT evaluation is active for the EMA workbench (8,000+ runs)

# 3. EMA Workbench and SALib imports
from ema_workbench import (
    Model,
    RealParameter,
    CategoricalParameter,
    ScalarOutcome,
    SequentialEvaluator,
    perform_experiments,
    Samplers,
)
try:
    from ema_workbench import Policy
except ImportError:
    try:
        from ema_workbench.em_framework.parameters import Policy
    except ImportError:
        from ema_workbench import Sample as Policy  # EMA Workbench 3.x
from ema_workbench.analysis import feature_scoring, prim
from ema_workbench.em_framework.salib_samplers import get_SALib_problem
from SALib.analyze import morris as morris_analyze
from SALib.analyze import sobol as sobol_analyze

import time as _time; _NB_START = _time.perf_counter()

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

print(f"Project root: {PROJECT_ROOT}")
print(f"Code directory exists: {CODE_DIR.exists()}")
print("EMA Workbench ready.")
print(
    f"{len(PERTURBABLE_PARAMS)} economic nuisance parameters + "
    f"{len(p.STRUCTURAL_UNCERTAINTIES)} structural ({', '.join(p.STRUCTURAL_UNCERTAINTIES.keys())}) = "
    f"{len(PERTURBABLE_PARAMS) + len(p.STRUCTURAL_UNCERTAINTIES)} total uncertainty dimensions available."
)



# Part 1 — Recap: what are `u_demand` and `u_beta_pt`?

In Notebook 02, all calculations were strictly deterministic because we held all uncertain inputs at their median trend ($u = 0.5$). 

To explore deep uncertainty, we parameterize the two major structural drivers using uniform draws $u \in [0, 1]$, which are converted via the inverse Normal CDF into 40-year trajectories:

1. **`u_demand` (Macro Demographic Force)**: Drives the **cumulative demand growth trajectory** (`m.demand_growth_path(u)`). It captures regional population and economic expansion over the 40-year horizon.
2. **`u_beta_pt` (Societal / Behavioral Preference)**: Drives the **public transport affinity trajectory** (`m.pt_affinity_path(u)`). It captures long-term cultural adoption of transit (e.g. climate awareness, GA pass ownership, telecommuting habits vs car dependency).

* **$u = 0.5$ (The Median Trend)**: Reproduces the exact mean baseline with zero deviation (used in Notebook 02).
* **$u < 0.5$ (Pessimistic Shift)**: Stagnant demand or declining transit adoption.
* **$u > 0.5$ (Optimistic Shift)**: Accelerated regional demand growth or strong cultural shift toward rail.

This notebook switches the EMA Workbench machinery on to sample these drivers across their full uncertainty space.


In [ ]:

# Recap: what Notebook 02 assumed (u = 0.5), and what different uncertainty draws look like
print("📌 Notebook 02 always used u = 0.5 (the median scenario, no uncertainty draw):")
print(f"  • Demand growth (g_cum) at u=0.5: Year 1 = {m.demand_growth_path(0.5)[0]:.1%}   Year 40 = {m.demand_growth_path(0.5)[-1]:.1%}")
print(f"  • PT affinity factor    at u=0.5: Year 1 = {m.pt_affinity_path(0.5)[0]:.3f}   Year 40 = {m.pt_affinity_path(0.5)[-1]:.3f}")

print("\n🎲 A different u ∈ [0, 1] yields a different 40-year trajectory:")
print("--- Demand Growth Trajectories (u_demand) ---")
for label, u in [("Pessimistic low growth", 0.1), ("Optimistic high growth", 0.9)]:
    traj = m.demand_growth_path(u)
    print(f"  u_demand={u} ({label:<24}): Year 1 = {traj[0]:.1%}   Year 40 = {traj[-1]:.1%}")

print("\n--- Public Transport Affinity Trajectories (u_beta_pt) ---")
for label, u in [("Car-heavy / low PT shift", 0.1), ("Strong transit adoption", 0.9)]:
    traj = m.pt_affinity_path(u)
    print(f"  u_beta_pt={u} ({label:<24}): Year 1 = {traj[0]:.3f}   Year 40 = {traj[-1]:.3f}")


# Part 2 — Define and register the uncertain parameters

Deep uncertainty means we do not have a single reliable probability distribution over future states of the world. Instead of predicting one "most likely future," we explore a wide space of plausible futures using the **EMA Workbench**.

In our model, parameters are split into three distinct registries in `code/parameters.py`:

1. **Structural Deep Uncertainties (`STRUCTURAL_UNCERTAINTIES`)**:
   - `u_demand`: Macroeconomic and demographic demand growth over 40 years.
   - `u_beta_pt`: Long-term cultural shift toward public transit adoption.
   - *Behavior:* Evolve over time with standard deviations widening from Year 1 to Year 40.
2. **Economic & Environmental Nuisance Uncertainties (`PERTURBABLE_PARAMS`)**:
   - Valuation parameters and unit costs (`C_FUEL`, `P_CO2`, `C_CO2`, `C_TT_CAR`, `C_TT_PT`, `DISCOUNT_RATE`, etc.).
   - *Behavior:* Sampled as $\text{Normal}(\text{nominal}, \pm 10\%)$ noise, held constant across the 40 years within each scenario.
3. **Fixed Engineering Design Facts (`FIXED_PARAMS`)**:
   - Infrastructure properties (`TUNNEL_TT_SAVING`, `FREQ_WAIT_SAVING`, `V_CAR`, `T_ACCESS`).
   - *Behavior:* Held strictly at their nominal engineering values (not sampled).

---

### Registering Uncertainties in EMA Workbench
Because our architecture is modular, calling `m.get_ema_uncertainties()` automatically constructs all `RealParameter(..., 0.0, 1.0)` objects directly from `parameters.py`.


In [ ]:

# 1. Inspect Fixed vs Perturbable parameter registries
print(f"📌 FIXED_PARAMS ({len(p.FIXED_PARAMS)}) — Physical/design facts (never perturbed):")
print("  " + ", ".join(p.FIXED_PARAMS.keys()))

print(f"\n🎲 PERTURBABLE_PARAMS ({len(p.PERTURBABLE_PARAMS)}) — Economic/environmental nuisance uncertainties (sampled ±10%):")
print("  " + ", ".join(p.PERTURBABLE_PARAMS.keys()))

print(f"\n📈 STRUCTURAL_UNCERTAINTIES ({len(p.STRUCTURAL_UNCERTAINTIES)}) — 40-year deep trajectories:")
for k, spec in p.STRUCTURAL_UNCERTAINTIES.items():
    print(f"  • {k:<12}: {spec['name']} ({spec['description']})")

# 2. Automatically register EMA Workbench RealParameter objects
ema_uncertainties = m.get_ema_uncertainties(include_nuisance=True)

print(f"\n✅ Successfully registered {len(ema_uncertainties)} EMA Workbench uncertainties:")
print(f"  • Structural ({len(p.STRUCTURAL_UNCERTAINTIES)}): {[u.name for u in ema_uncertainties[:len(p.STRUCTURAL_UNCERTAINTIES)]]}")
print(f"  • Nuisance ({len(p.PERTURBABLE_PARAMS)}): {[u.name for u in ema_uncertainties[len(p.STRUCTURAL_UNCERTAINTIES):len(p.STRUCTURAL_UNCERTAINTIES)+5]]} ... (+{len(p.PERTURBABLE_PARAMS)-5} more)")


### How uncertainty grows between Year 1 and Year 40

Both structural uncertainties follow the same widening cone principle: a **Normal distribution** whose mean follows the nominal baseline trend line and whose **standard deviation widens over time** — tight today ($\sigma_1$), and much wider by Year 40 ($\sigma_{40}$).

From `code/parameters.py`:

| Structural Parameter | Year 1 Mean | Year 1 $\sigma_1$ | Year 40 Mean | Year 40 $\sigma_{40}$ | Interpretation |
|---|---|---|---|---|---|
| **Demand Growth (`u_demand`)** | $0.0\%$ | $\pm 0.5\%$ | $+31.2\%$ | $\pm 30.0\%$ | Regional demographic & macroeconomic expansion |
| **PT Preference (`u_beta_pt`)** | $1.00$ | $\pm 0.01$ | $1.20$ | $\pm 0.25$ | Cultural adoption of public transit (multiplier) |

**The Widening Rule**: Today's snapshot has little uncertainty ($\sigma_1 \approx 0.5\% - 1\%$), while four decades into the future, the uncertainty cone opens widely ($\sigma_{40} \approx 15\% - 20\%$).

Both are transformed dynamically in `simulation_engine.py`: a uniform draw $u \in [0, 1]$ is converted via the inverse Normal CDF ($z = \Phi^{-1}(u)$) into a shock:
$$\text{Trajectory: } \text{value}(t) = \text{mean}(t) + z \cdot \sigma(t)$$


In [ ]:
# Build the widening-sigma table dynamically from STRUCTURAL_UNCERTAINTIES
rows = []

for u_key, spec in p.STRUCTURAL_UNCERTAINTIES.items():
    rows.append({
        "Uncertainty Key": u_key,
        "Description": spec["name"],
        "Year 1 Mean": spec["nominal_start"],
        "Year 1 Sigma (σ1)": spec["sigma_start"],
        "Year 40 Mean": spec["nominal_end"],
        "Year 40 Sigma (σ40)": spec["sigma_end"],
    })

sigma_table = pd.DataFrame(rows).set_index("Uncertainty Key")

print("📊 Structural Uncertainty Specifications (Year 1 vs. Year 40):")
display(sigma_table.style.format({
    "Year 1 Mean": "{:.2f}",
    "Year 1 Sigma (σ1)": "{:.3f}",
    "Year 40 Mean": "{:.2f}",
    "Year 40 Sigma (σ40)": "{:.3f}",
}))


## Visualising Year 1 vs. Year 40 Uncertainty Distributions

To see how deep uncertainty widens over time, we draw $N = 3{,}000$ Monte Carlo samples from $u \sim \text{Uniform}[0, 1]$ and convert each draw into its **Year 1** (today) and **Year 40** (future) snapshot values for our two structural uncertainties (`u_demand` and `u_beta_pt`).

The hybrid violin + boxplots below illustrate the fundamental nature of deep planning horizons:
* **Year 1 (Today)**: The distribution is a tight, narrow sliver because current demographic growth and commuting preferences are well-calibrated with minimal variance ($\sigma_1 \approx 0.5\% - 1\%$).
* **Year 40 (Future)**: The distribution expands into a broad uncertainty cone ($\sigma_{40} \approx 15\% - 20\%$), capturing the wide range of plausible long-term futures that infrastructure planners must prepare for.


In [ ]:
# Sample N_DIST uniform draws and evaluate Year 1 and Year 40 values
N_DIST = 3000
rng = np.random.default_rng(0)

u_demand_draws = rng.uniform(0, 1, N_DIST)
u_pt_draws     = rng.uniform(0, 1, N_DIST)

# Convert uniform draws to physical trajectory values
dg_y1  = np.array([m.demand_growth_path(u)[0]  for u in u_demand_draws])
dg_y40 = np.array([m.demand_growth_path(u)[-1] for u in u_demand_draws])

pt_y1  = np.array([m.pt_affinity_path(u)[0]    for u in u_pt_draws])
pt_y40 = np.array([m.pt_affinity_path(u)[-1]   for u in u_pt_draws])

df_demand = pd.DataFrame({
    "value": np.concatenate([dg_y1, dg_y40]),
    "year":  ["Year 1"] * N_DIST + ["Year 40"] * N_DIST,
})

df_pt = pd.DataFrame({
    "value": np.concatenate([pt_y1, pt_y40]),
    "year":  ["Year 1"] * N_DIST + ["Year 40"] * N_DIST,
})

# Plot Side-by-Side Violin + Boxplots
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

plot_specs = [
    (df_demand, "Cumulative Demand Growth (g_cum)", "g_cum (fraction / %)", axes[0]),
    (df_pt, "Public Transport Preference (u_beta_pt)", "PT Affinity Multiplier", axes[1]),
]

for df_data, title, ylabel, ax in plot_specs:
    # 1. Violin density plot
    sns.violinplot(
        data=df_data, x="year", y="value", ax=ax, inner=None,
        color="#a6bddb", cut=0, linewidth=1.2
    )
    # 2. Overlay boxplot for quartiles and median
    sns.boxplot(
        data=df_data, x="year", y="value", ax=ax, width=0.15, showfliers=False,
        boxprops={"facecolor": "white", "zorder": 3, "edgecolor": "#333"},
        whiskerprops={"zorder": 3, "color": "#333"},
        capprops={"zorder": 3, "color": "#333"},
        medianprops={"color": "#e41a1c", "linewidth": 2.2, "zorder": 4},
        zorder=3
    )
    
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.suptitle(f"Year 1 vs. Year 40 Uncertainty Distributions ({N_DIST:,} Monte Carlo draws each)", y=1.02, fontsize=12, fontweight="bold")
plt.tight_layout()

# Save figure to figures directory
figures_path = FIGURES_DIR / "03_year1_vs_year40_violin.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Figure saved to {figures_path}")
print("📌 Notice: The Year 1 violin is a tight sliver (little disagreement about today),")
print("   while Year 40 opens widely into an uncertainty cone. That widening IS the deep uncertainty explored in this notebook.")


## Register the uncertainties with EMA Workbench

The model wrapper below (`mehrspur_snapshot_model`) is what the **EMA Workbench** invokes once per experimental scenario. 

It takes `u_demand`, `u_beta_pt`, a static infrastructure `pathway` key, and arbitrary `u_<PARAM>` nuisance draws. It returns **both** the Year-1 and Year-40 snapshot for every physical indicator:
* Modal shares (`car_share`, `pt_share`)
* Average travel time (`avg_travel_time_min`)
* Bottleneck congestion delay (`congestion_delay_hours`)
* Direct tailpipe emissions (`co2_tonnes`)
* Total passenger trips (`total_demand`)


In [ ]:
import stages as stages_module

# 1. Initialize transport context if not already loaded
if "ctx" not in globals():
    print("Loading Canton Zürich transport model context...")
    ctx = tmi.load_transport_context(PROJECT_ROOT)
corridor_zones = tmi.get_zone_ids_for_municipalities(ctx, p.CORRIDOR_MUNICIPALITIES)
stages = stages_module.get_stages(p.NOMINAL_PARAMS)
# Pre-cache baseline corridor metrics and mode results for Stage 0, 1, and 2
STAGE_METRICS = {}
MODE_RESULTS = {}
for s in [0, 1, 2]:
    res, metrics = tmi.run_simulation(
        ctx, 
        stage=s, 
        stage_specs=stages, 
        corridor_zone_ids=corridor_zones,
        corridor_municipalities=p.CORRIDOR_MUNICIPALITIES
    )
    STAGE_METRICS[s] = metrics
    MODE_RESULTS[s] = res

# Static pathways mapped to active stage
STATIC_PATHWAYS = ["baseline", "static1", "static2"]
PATHWAY_STAGE = {
    "baseline": 0,  # Stage 0 for all 40 years
    "static1":   1,  # Stage 1 for all 40 years
    "static2":   2,  # Stage 2 for all 40 years
}

def evaluate_snapshot(stage: int, pt_affinity: float, year_idx: int, g_cum: float, params: dict) -> dict:
    """One Year-1 or Year-40 snapshot: physical indicators for a fixed stage."""
    base_metrics = STAGE_METRICS[stage]
    mode_res = MODE_RESULTS.get(stage)
    
    r = m.simulate_year(
        base_metrics, 
        stage=stage, 
        year_idx=year_idx, 
        g_cum=g_cum, 
        params=params,
        context=ctx,
        mode_result=mode_res,
        corridor_municipalities=p.CORRIDOR_MUNICIPALITIES
    )
    
    base_pt = base_metrics.get("pt_share", 0.0)
    base_car = base_metrics.get("car_share", 0.0)
    
    new_pt = base_pt * pt_affinity
    delta_pt = new_pt - base_pt
    new_car = max(base_car - delta_pt, 0.0)
    
    p2a = params.get("PEAK_TO_ANNUAL", getattr(p, "PEAK_TO_ANNUAL", 1200.0))
    return {
        "car_share":               new_car,
        "pt_share":                new_pt,
        "avg_travel_time_min":     r["avg_tt_min"],
        "congestion_delay_hours":  r["congestion_delay_hours"] / p2a,
        "co2_tonnes":              r["co2_tonnes"] / p2a,
        "total_demand":            r["total_demand"] / p2a,
    }




def mehrspur_snapshot_model(u_demand=0.5, u_beta_pt=0.5, pathway="baseline", **nuisance_u):
    """EMA Workbench wrapper: returns every physical outcome for BOTH the Year-1 and Year-40 snapshot."""
    params = m.sample_perturbed_params(nuisance_u)
    stage = PATHWAY_STAGE.get(pathway, 0)
    g_path  = m.demand_growth_path(u_demand)
    pt_path = m.pt_affinity_path(u_beta_pt)

    y1  = evaluate_snapshot(stage, pt_path[0],  year_idx=0,           g_cum=g_path[0],  params=params)
    y40 = evaluate_snapshot(stage, pt_path[-1], year_idx=p.N_YEARS-1, g_cum=g_path[-1], params=params)

    out = {}
    for key, value in y1.items():
        out[f"{key}_y1"] = value
    for key, value in y40.items():
        out[f"{key}_y40"] = value
    return out

OUTCOME_BASE = ["car_share", "pt_share", "avg_travel_time_min", "congestion_delay_hours", "co2_tonnes", "total_demand"]
OUTCOMES = [f"{base}_{year}" for base in OUTCOME_BASE for year in ("y1", "y40")]

print(f"✅ Registered {len(OUTCOMES)} outcomes ({len(OUTCOME_BASE)} physical indicators x 2 snapshot years):")
print(OUTCOMES)


### The 2-Uncertainty Model (Structural Deep Drivers)

First, we configure the model with only our two structural macro uncertainties (`u_demand` and `u_beta_pt`). All economic nuisance parameters remain at their default nominal values ($u = 0.5$).


In [ ]:
ema_model_2d = Model("MehrSpur2D", function=mehrspur_snapshot_model)
ema_model_2d.uncertainties = [
    RealParameter("u_demand", 0.0, 1.0),
    RealParameter("u_beta_pt", 0.0, 1.0),
]
ema_model_2d.outcomes = [ScalarOutcome(o) for o in OUTCOMES]

print(f"✅ Core 2D Model: {len(ema_model_2d.uncertainties)} uncertainties registered:",
      [u.name for u in ema_model_2d.uncertainties])


### The Full-Dimensional Uncertainty Model (Structural + Economic Shocks)

Next, we register all economic and environmental nuisance parameters (`PERTURBABLE_PARAMS`) alongside the structural drivers, creating a full-dimensional exploratory model.


In [ ]:
ema_model_full = Model("MehrSpurFull", function=mehrspur_snapshot_model)
ema_model_full.uncertainties = m.get_ema_uncertainties(include_nuisance=True)
ema_model_full.outcomes = [ScalarOutcome(o) for o in OUTCOMES]
U_COLS_FULL = [u.name for u in ema_model_full.uncertainties]

print(f"✅ Full Model: {len(ema_model_full.uncertainties)} uncertainties registered "
      f"(2 structural + {len(p.PERTURBABLE_PARAMS)} nuisance parameters).")


## Sampling the Uncertainty Space

Knowing which parameters are uncertain is not enough — we must **sample** them efficiently to explore the space of plausible futures without running millions of simulations. The EMA Workbench supports several experimental designs:

* **Monte Carlo (MC)**: Independent random uniform draws. Simple, but can leave un-sampled gaps or create redundant clusters, especially with a low number of runs.
* **Latin Hypercube Sampling (LHS)**: Stratifies each parameter into equal-probability bins and guarantees even marginal coverage along every single axis. Usually the preferred default for exploratory modeling.
* **Sobol**: A quasi-random, highly structured grid design built specifically for **variance-based global sensitivity analysis**. Because it isolates the specific variance contribution of each parameter, it requires a strict number of runs: $N \times (2D + 2)$, where $D$ is the number of uncertainties and $N$ is the number of base points (ideally a power of two).

Below, we run a short 400-scenario experiment on our 2D core model to visually compare how these three algorithms cover the parameter space.


In [ ]:


N_SCENARIOS = 200
sampling_results_2d = {}

# 1. Run Monte Carlo and Latin Hypercube
for label, sampler in [("Monte Carlo", Samplers.MC), ("Latin Hypercube", Samplers.LHS)]:
    with SequentialEvaluator(ema_model_2d) as evaluator:
        experiments, outcomes = perform_experiments(
            ema_model_2d, N_SCENARIOS, 
            evaluator=evaluator, uncertainty_sampling=sampler, log_progress=False
        )
    df = experiments.copy()
    for col, arr in outcomes.items():
        df[col] = arr
    sampling_results_2d[label] = df

# 2. Run Sobol (Requires exactly N * (2D + 2) runs)
# For D=2 structural uncertainties, N=64 yields 64 * (2*2 + 2) = 384 runs
N_SOBOL_2D = 64
with SequentialEvaluator(ema_model_2d) as evaluator:
    experiments, outcomes = perform_experiments(
        ema_model_2d, N_SOBOL_2D, 
        evaluator=evaluator, uncertainty_sampling=Samplers.SOBOL, log_progress=False
    )
df_sobol_2d = experiments.copy()
for col, arr in outcomes.items():
    df_sobol_2d[col] = arr
sampling_results_2d["Sobol"] = df_sobol_2d

# 3. Print counts and visualize the 2D coverage
print("Simulation Runs Generated:")
for label, df in sampling_results_2d.items():
    print(f"  • {label:<16}: {len(df)} scenarios")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), sharex=True, sharey=True)
colors = {"Monte Carlo": "#d95f02", "Latin Hypercube": "#2b5c8f", "Sobol": "#1b9e77"}

for ax, (label, df) in zip(axes, sampling_results_2d.items()):
    ax.scatter(df["u_demand"], df["u_beta_pt"], s=20, alpha=0.7, color=colors[label], edgecolors="none")
    ax.set_title(f"{label}  (n={len(df)})", fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Demand Growth (u_demand)")
    ax.grid(alpha=0.3, linestyle=":")
    
axes[0].set_ylabel("PT Preference (u_beta_pt)")

plt.suptitle("Coverage of the 2-D Uncertainty Space by Sampling Method", y=1.03, fontsize=12, fontweight="bold")
plt.tight_layout()

# Save figure
figures_path = FIGURES_DIR / "03_sampling_coverage_2d.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()

print("📌 Notice: Monte Carlo leaves random gaps, LHS spreads points beautifully, and Sobol generates a highly structured grid.")


### Tracking Convergence to the Mean (PT Share)

A sampling design covering the space visually is one thing; whether its **estimate of the mean** has actually stabilized is another. The chart below tracks the running mean of `pt_share_y40` as more scenarios are evaluated. 

Notice how the structured samplers (LHS and Sobol) often stabilize around the true mean faster and with less jitter than purely random Monte Carlo draws.


In [ ]:
def running_mean(series):
    return series.expanding().mean()

fig, ax = plt.subplots(figsize=(9, 4.5))

# 1. Plot running means for all samplers
for label, color in colors.items():
    df = sampling_results_2d[label]
    
    ax.plot(
        range(1, len(df) + 1),
        running_mean(df["pt_share_y40"]),
        color=color,
        linewidth=2,
        label=f"{label} (n={len(df)})",
    )

# 2. Establish a "true" reference mean combining all generated scenarios
all_results = pd.concat(
    [df["pt_share_y40"] for df in sampling_results_2d.values()],
    ignore_index=True,
)
reference_mean = all_results.mean()

ax.axhline(
    reference_mean,
    color="grey",
    linestyle="--",
    linewidth=1.5,
    label=f"Combined Reference Mean ({reference_mean:.1%})"
)

# 3. Formatting
ax.set_xlabel("Number of scenarios evaluated", fontsize=10)
ax.set_ylabel("Running Mean (Year-40 PT Share)", fontsize=10)
ax.set_title("Convergence of mean Year-40 PT share by sampling method\n(2D Uncertainty Space)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9, loc="lower right", frameon=True)
ax.grid(alpha=0.3, linestyle=":")

# Formatter for percentage y-axis
import matplotlib.ticker as mtick
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))

plt.tight_layout()
figures_path = FIGURES_DIR / "03_sampling_convergence_2d.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()


### Convergence in the Full 16-Dimensional Space

Let's repeat the convergence exercise on `ema_model_full`, which includes our 2 structural drivers *plus* 14 economic/nuisance parameters. Two things to watch for:

1. **Sobol gets expensive fast.** With $D=16$ uncertainties, the standard second-order Sobol design requires $N \times (2D + 2) = N \times 34$ runs. To keep our computational budget around 1,000 runs for this demonstration, we use $N=32$ base points (the nearest power of two), resulting in exactly **1,088 model evaluations**. For a true, robust Sobol analysis, $N$ often needs to be 512 or 1024, which would require over 17,000 runs — this is why Part 5 introduces screening methods to eliminate non-influential parameters first!
2. **MC and LHS take longer to "settle".** In a 16-dimensional space, there is simply more room for a 400-point sample to be unrepresentative of the true mean compared to a 2D space.


In [ ]:
sampling_results_full = {}

# 1. Run Monte Carlo and Latin Hypercube
for label, sampler in [("Monte Carlo", Samplers.MC), ("Latin Hypercube", Samplers.LHS)]:
    with SequentialEvaluator(ema_model_full) as evaluator:
        experiments, outcomes = perform_experiments(
            ema_model_full, N_SCENARIOS, 
            evaluator=evaluator, uncertainty_sampling=sampler, log_progress=False
        )
    df = experiments.copy()
    for col, arr in outcomes.items():
        df[col] = arr
    sampling_results_full[label] = df

# 2. Run Sobol
# D=16 -> N * (2*16 + 2) = N * 34 runs. 
# We use N=32 base points for 32 * 34 = 1,088 runs purely for visual illustration.
N_SOBOL_FULL = 32
with SequentialEvaluator(ema_model_full) as evaluator:
    experiments, outcomes = perform_experiments(
        ema_model_full, N_SOBOL_FULL, 
        evaluator=evaluator, uncertainty_sampling=Samplers.SOBOL, log_progress=False
    )
df_sobol_full = experiments.copy()
for col, arr in outcomes.items():
    df_sobol_full[col] = arr
sampling_results_full["Sobol"] = df_sobol_full

# 3. Print design summary
n_uncertainties = len(ema_model_full.uncertainties)
print(f"D = {n_uncertainties} uncertainties.")
print(f"Sobol design used here: N={N_SOBOL_FULL} base points x {2 * n_uncertainties + 2} = {len(df_sobol_full)} runs.")
print(f"For a robust analysis, N=512 would require {512 * (2 * n_uncertainties + 2):,} runs for a single policy!")

# 4. Visualize convergence
fig, ax = plt.subplots(figsize=(9, 4.5))

for label, color in colors.items():
    df = sampling_results_full[label]
    
    ax.plot(
        range(1, len(df) + 1),
        running_mean(df["pt_share_y40"]),
        color=color,
        linewidth=2,
        label=f"{label} (n={len(df)})",
    )

all_results = pd.concat(
    [df["pt_share_y40"] for df in sampling_results_full.values()],
    ignore_index=True,
)
reference_mean = all_results.mean()

ax.axhline(
    reference_mean,
    color="grey",
    linestyle="--",
    linewidth=1.5,
    label=f"Combined Reference Mean ({reference_mean:.1%})"
)

ax.set_xlabel("Number of scenarios evaluated", fontsize=10)
ax.set_ylabel("Running Mean (Year-40 PT Share)", fontsize=10)
ax.set_title(f"Convergence of mean Year-40 PT share by sampling method\n({n_uncertainties}-D Uncertainty Space)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9, loc="lower right", frameon=True)
ax.grid(alpha=0.3, linestyle=":")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))

plt.tight_layout()
figures_path = FIGURES_DIR / "03_sampling_convergence_full.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 1. Project the 16D design back onto the 2D structural plane
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharex=True, sharey=True)

for ax, label in zip(axes, colors.keys()):
    simple = sampling_results_2d[label]
    full = sampling_results_full[label]

    # Plot the original 2D design in grey shadow
    ax.scatter(simple["u_demand"], simple["u_beta_pt"],
               s=20, alpha=0.35, color="grey", edgecolors="none", label="2D Design")
    
    # Overlay the new 16D design on top
    ax.scatter(full["u_demand"], full["u_beta_pt"],
               s=18, alpha=0.8, color=colors[label], edgecolors="none", label="16D Design")

    ax.set_title(f"{label}\n(2D: n={len(simple):,} | 16D: n={len(full):,})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Demand Growth (u_demand)")
    ax.grid(alpha=0.3, linestyle=":")

axes[0].set_ylabel("PT Preference (u_beta_pt)")
axes[0].legend(fontsize=9, loc="best", frameon=True)

plt.suptitle(
    f"Coverage of Structural Uncertainties: 2D versus {n_uncertainties}D Design\n"
    f"Grey = original 2D points; Colors = new {n_uncertainties}D points projected onto the 2D plane",
    y=1.08, fontsize=11, fontweight="bold"
)

plt.tight_layout()
figures_path = FIGURES_DIR / "03_sampling_coverage_full.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()

print("📌 Notice: While LHS guarantees even coverage when projected down to 1D, when projected down to 2D from 16D, it looks essentially random (like Monte Carlo). The Sobol grid, however, maintains distinct structured clustering even when projected from a high-dimensional space.")


### Quantifying Coverage and Discrepancy

Although the full Sobol design contains more model evaluations, many points appear to perfectly overlap in the 2D projection above. This is a mathematical feature, not a bug: these evaluations share the exact same `u_demand` and `u_beta_pt` values, but differ along one or more of the other 14 nuisance dimensions. They are highly distinct points in the complete 16-dimensional space.

To objectively measure how well these samplers spread their points, we can compute two metrics:
1. **Discrepancy (lower is better)**: Measures how much the empirical distribution of points deviates from a perfectly uniform distribution.
2. **Mean Pairwise Coverage (higher is better)**: Grids the space into bins and calculates what percentage of possible 2D coordinate bins contain at least one sample, averaged across all possible pairs of dimensions.


In [ ]:
from itertools import combinations
from scipy.stats import qmc

def grid_coverage(X, bins=10):
    """Calculates the percentage of bins occupied by at least one sample."""
    cells = np.floor(X * bins).clip(0, bins - 1).astype(int)
    return len(np.unique(cells, axis=0)) / (bins ** X.shape[1])

rows = []

for method, df in sampling_results_full.items():
    X_full = df[U_COLS_FULL].to_numpy()
    X_2 = df[["u_demand", "u_beta_pt"]].to_numpy()

    # Calculate coverage for all possible 2D pairs in the 16D space
    pair_coverages = [
        grid_coverage(X_full[:, [i, j]])
        for i, j in combinations(range(X_full.shape[1]), 2)
    ]

    rows.append({
        "method": method,
        "n": len(df),
        "16D_discrepancy": qmc.discrepancy(X_full),
        "16D_coverage": np.mean(pair_coverages),
        "2D_discrepancy": qmc.discrepancy(X_2),
        "2D_coverage": grid_coverage(X_2),
    })

comparison = pd.DataFrame(rows).set_index("method")
display(comparison.round(4))

# ---------------------------------------------------------
# Plotting the metrics
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))

metrics = {
    "16D_discrepancy": f"{n_uncertainties}D Discrepancy\n(Lower is better)",
    "16D_coverage": "Mean Pairwise Coverage\n(Higher is better)",
    "2D_discrepancy": "Projected 2D Discrepancy\n(Lower is better)",
    "2D_coverage": "Projected 2D Coverage\n(Higher is better)",
}

for ax, (metric, title) in zip(axes, metrics.items()):
    # Discrepancy sorts ascending (lower is better); Coverage sorts descending (higher is better)
    ascending = "discrepancy" in metric
    order = comparison[metric].sort_values(ascending=ascending).index
    labels = [f"{x}\n(n={comparison.loc[x, 'n']:,})" for x in order]

    ax.bar(labels, comparison.loc[order, metric], color=[colors[x] for x in order])
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle=":")

plt.suptitle("Objective Coverage Metrics of the Full and Projected Uncertainty Spaces", y=1.05, fontsize=12, fontweight="bold")
plt.tight_layout()

figures_path = FIGURES_DIR / "03_sampling_coverage_metrics.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()

print("📌 Notice: LHS dominates in uniform coverage, while Sobol has the lowest discrepancy. Monte Carlo is consistently the weakest at evenly exploring high-dimensional spaces.")


---

### Establishing the Working Dataset

From here on, we use the **full 16-dimensional sample** (`ema_model_full`, Latin Hypercube, 400 scenarios) as the realistic uncertainty space for the rest of this notebook. Rather than regenerating scenarios for every analysis, we lock it in as a single, reusable dataset (`df_baseline`).


In [ ]:
df_baseline = sampling_results_full["Latin Hypercube"].copy()
print(f"✅ Working dataset established: {len(df_baseline)} scenarios x {n_uncertainties} uncertainties")
print(f"   (baseline pathway, Year 1 + Year 40 snapshots).")
# Quick statistical summary
df_baseline[["u_demand", "u_beta_pt"] + OUTCOMES].describe().round(3)

### Visual Overview of the Baseline Dataset

The histograms below show the distributions of our two structural uncertainty inputs alongside the resulting Year-1 and Year-40 outcomes. 

* The **dashed orange line** indicates the mean.
* The **dotted black line** indicates the median.
* Differences between them highlight skewed distributions, while the width of the bell curve shows how much variance exists across the 400 plausible futures.


In [ ]:
plot_cols = ["u_demand", "u_beta_pt"] + OUTCOMES
n_cols = 5
n_rows = int(np.ceil(len(plot_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 2.8 * n_rows))
axes = np.asarray(axes).ravel()

for ax, col in zip(axes, plot_cols):
    values = df_baseline[col].dropna()

    counts, bins, _ = ax.hist(
        values, bins=20, density=True,
        color="steelblue", alpha=0.45, edgecolor="white",
    )

    ax.axvline(values.mean(), color="darkorange", linestyle="--", linewidth=1.3, label="Mean")
    ax.axvline(values.median(), color="black", linestyle=":", linewidth=1.3, label="Median")

    # 1. Clean title formatting
    ax.set_title(col.replace("_", " ").title(), fontsize=10, fontweight="bold")
    
    # 2. Percentage formatting on x-axis for modal share columns
    if "share" in col:
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
    
    # 3. Smart x-axis limits to prevent collapsing if variance is narrow
    if values.std() < 1e-4:
        val = values.mean()
        ax.set_xlim(max(0.0, val - 0.05), min(1.0, val + 0.05))

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(alpha=0.2, linestyle=":")

# Remove empty axes
for ax in axes[len(plot_cols):]:
    ax.remove()

axes[0].legend(fontsize=8, loc="best", frameon=True)

plt.suptitle("Distributions of Baseline Uncertainties and Outcomes", y=1.02, fontsize=12, fontweight="bold")
plt.tight_layout()

figures_path = FIGURES_DIR / "03_baseline_distributions.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()


### Mapping Outcomes onto the Uncertainty Space

The figure below projects the 16-dimensional Latin Hypercube design back onto the two structural axes, but this time **colored by the resulting Year-40 outcome**. 

* Smooth color gradients indicate that `u_demand` or `u_beta_pt` systematically influence the outcome. 
* Considerable color noise/scatter among nearby points indicates that the *other* 14 nuisance parameters also play a heavy role in determining the final outcome.


In [ ]:
lhs_final = sampling_results_full["Latin Hypercube"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

plots = [
    ("congestion_delay_hours_y40", "Scenario Space colored by Year-40 Congestion", "Congestion Delay (Hours)", "RdYlGn_r"),
    ("pt_share_y40", "Scenario Space colored by Year-40 PT Share", "Public Transport Share", "Blues"),
]

for ax, (outcome, title, cbar_label, cmap) in zip(axes, plots):
    sc = ax.scatter(
        lhs_final["u_demand"], lhs_final["u_beta_pt"],
        c=lhs_final[outcome], cmap=cmap, s=35,
        alpha=0.85, edgecolors="none"
    )
    
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("Demand Growth (u_demand)", fontsize=9)
    ax.grid(alpha=0.3, linestyle=":")
    
    cbar = fig.colorbar(sc, ax=ax, pad=0.04)
    cbar.set_label(cbar_label, fontsize=9)
    
    if "share" in outcome:
        # Format colorbar as percentage
        cbar.ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))

axes[0].set_ylabel("PT Preference (u_beta_pt)", fontsize=9)

plt.suptitle("Final Latin Hypercube Design — Outcomes across the Uncertainty Space", y=1.03, fontsize=12, fontweight="bold")
plt.tight_layout()

figures_path = FIGURES_DIR / "03_lhs_scenario_space_outcomes.png"
plt.savefig(figures_path, dpi=150, bbox_inches="tight")
plt.show()


## Part 4 — Explore the Outcome Space

*"What kinds of futures are actually possible in our baseline scenario?"*

Before we use algorithms to figure out *which* uncertainty matters most, we should look at the raw statistical spread of our physical outcomes (modal split, travel time, congestion, emissions). 

By comparing Year 1 and Year 40 side by side in the **baseline (do-nothing)** pathway, we can clearly see how the *range* of plausible outcomes severely widens as our planning horizon extends into the future.


In [ ]:
# Create a cleanly sorted list of column pairs (Year 1 vs Year 40 for each indicator)
summary_cols = [f"{base}_{yr}" for base in OUTCOME_BASE for yr in ("y1", "y40")]

# Display transposed summary statistics to easily compare the widening spread
display(df_baseline[summary_cols].describe().T.round(3))


## 4.2 Distributional parallel-coordinates view — Year 1 vs. Year 40

A traditional parallel-coordinates plot draws one line per scenario, which quickly becomes illegible across hundreds of scenarios and fails to communicate underlying probabilities. 

Instead, each vertical axis below displays a **split violin plot**:
- **Left half (Blue)**: Outcome distribution at **Year 1**.
- **Right half (Orange)**: Outcome distribution at **Year 40**.

Every indicator is min-max normalized to $[0, 1]$ (pooled across both years) to bring indicators with different units (percentages, minutes, delay hours, tonnes) onto a unified visual scale:
- **Non-overlapping halves**: The 40-year horizon trend dominates the outcome (e.g., PT share shifting upwards with behavioral adoption, and congestion delay surging under demand growth).
- **Heavily overlapping halves**: Near-term Year-1 uncertainty already accounts for most of the plausible 40-year range.


In [ ]:
import seaborn as sns

# Active SBB MehrSpur outcome indicators from the EMA experiment
PARCOORD_OBJS = [
    "car_share",
    "pt_share",
    "avg_travel_time_min",
    "congestion_delay_hours",
    "co2_tonnes",
    "total_demand",
]

PARCOORD_LABELS = {
    "car_share": "Car Share",
    "pt_share": "PT Share",
    "avg_travel_time_min": "Avg Travel\nTime (min)",
    "congestion_delay_hours": "Congestion\nDelay (hrs)",
    "co2_tonnes": "CO₂ Emissions\n(tonnes)",
    "total_demand": "Total Peak\nTrips",
}

long_rows = []
for obj in PARCOORD_OBJS:
    if f"{obj}_y1" not in df_baseline.columns or f"{obj}_y40" not in df_baseline.columns:
        continue
    y1  = df_baseline[f"{obj}_y1"].values
    y40 = df_baseline[f"{obj}_y40"].values
    lo, hi = min(y1.min(), y40.min()), max(y1.max(), y40.max())
    span = (hi - lo) if hi > lo else 1.0
    for values, year in [(y1, "Year 1"), (y40, "Year 40")]:
        for v in (values - lo) / span:
            long_rows.append({"objective": PARCOORD_LABELS[obj], "year": year, "norm_value": v})

parcoord_df = pd.DataFrame(long_rows)

fig, ax = plt.subplots(figsize=(12, 5.5))
sns.violinplot(
    data=parcoord_df,
    x="objective",
    y="norm_value",
    hue="year",
    split=True,
    inner="quart",
    density_norm="width",
    common_norm=False,
    bw_adjust=0.8,
    cut=0,
    palette={"Year 1": "#2b5c8f", "Year 40": "#d95f02"},
    ax=ax,
)

ax.set_ylabel("Normalized Value (0 = min, 1 = max, pooled across both years)", fontweight="bold")
ax.set_xlabel("")
ax.set_title("Distributional Parallel Coordinates — Baseline Pathway (Year 1 vs. Year 40)", fontweight="bold", fontsize=12, pad=12)
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(title="Horizon", loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "03_parcoords_violin.png", dpi=150, bbox_inches="tight")
plt.show()

print("💡 Reading the chart:")
print("• A lane where the two halves barely overlap (e.g., PT share or Demand) is one where the 40-year macro trend does most of the work.")
print("• A lane with heavy overlap is one where Year-1 uncertainty already spans most of the plausible 40-year spread.")


# Part 5 — Which uncertainty matters most?

*"Which uncertain factors matter most?"*

With 16 candidate uncertainties (2 structural macro-drivers + 14 economic, environmental, and cost parameters), looking at individual scatter plots is no longer enough. We need a systematic way to rank them:

1. **Extra Trees feature scoring** — a fast, non-parametric screening pass across all 16 parameters (mandatory).
2. **Morris screening** — a structured, efficient method to narrow the parameters down to key drivers (advanced).
3. **Sobol $S_1$ / $S_T$ indices** — a formal, variance-based global sensitivity analysis on the prioritized subset (advanced).

**An important distinction:** Feature scoring (using an Extra-Trees regressor) evaluates *"how useful is this input for predicting the output variation"*, acting as a fast **proxy** for sensitivity. Sobol indices directly measure *"what fraction of the output's variance is explained by this input"*. Report Extra-Trees scores as **"normalised relative importance"**, not as "percentage of variance explained" (which is reserved for Sobol).

---

### Extra Trees feature scoring: Year 1 vs. Year 40

We score every one of the 16 uncertainties against `pt_share` (public transport mode share) — separately for the Year-1 and Year-40 snapshots — using the 16-D Latin Hypercube sample (`df_baseline`). The chart displays the top parameters as a **stacked bar** (normalised to 100%, with an "other" segment absorbing remaining parameters).


In [ ]:
# 1. Score all 16 uncertainties against pt_share in Year 1 and Year 40
x_full = df_baseline[U_COLS_FULL]
fs_y1 = feature_scoring.get_feature_scores_all(
    x_full, {"pt_share_y1": df_baseline["pt_share_y1"].values}
)
fs_y40 = feature_scoring.get_feature_scores_all(
    x_full, {"pt_share_y40": df_baseline["pt_share_y40"].values}
)

TOP_N = 8
combined_max = pd.concat(
    [fs_y1["pt_share_y1"], fs_y40["pt_share_y40"]], axis=1
).max(axis=1)
top_params = (
    combined_max.sort_values(ascending=False).head(TOP_N).index.tolist()
)

print(
    f"Feature scores sum to 1.0 by construction (Year 1: {fs_y1['pt_share_y1'].sum():.3f}, "
    f"Year 40: {fs_y40['pt_share_y40'].sum():.3f}) -- they are a normalised ranking, not a"
)
print("physical unit.")
print()
print(f"Top {TOP_N} parameters by peak importance across both snapshots:")
print(combined_max.loc[top_params].round(3).to_string())


In [ ]:
# Score against Congestion Delay in Year 40
fs_cong_y40 = feature_scoring.get_feature_scores_all(
    x_full,
    {"congestion_delay_hours_y40": df_baseline["congestion_delay_hours_y40"].values},
)
print("Top drivers of Year-40 Congestion Delay:")
print(
    fs_cong_y40["congestion_delay_hours_y40"]
    .sort_values(ascending=False)
    .head(5)
    .round(3)
)


In [ ]:
PARAMETER_LABELS = {
    "u_demand": "Travel-demand growth trajectory",
    "u_beta_pt": "PT preference trajectory",
    "u_DISCOUNT_RATE": "Social discount rate",
    "u_F_FUEL": "Car fuel consumption",
    "u_C_FUEL": "Fuel price",
    "u_P_CO2": "Car CO₂ emission factor",
    "u_C_CO2": "Carbon cost (base year)",
    "u_C_CO2_GROWTH": "Carbon-cost annual increase",
    "u_C_TT_CAR": "Value of car travel time",
    "u_C_TT_PT": "Value of PT travel time",
    "u_C_FARE": "Public transport fare",
    "u_C_INV_STAGE1": "Stage 1 capex (Local Stations & Access)",
    "u_C_OP_STAGE1": "Stage 1 opex",
    "u_C_INV_STAGE2": "Stage 2 capex (Core Tunnel)",
    "u_C_OP_STAGE2": "Stage 2 opex",
    "u_C_FLEX": "Flexibility option premium",
}


In [ ]:
def stacked_series(fs_col: pd.Series, top_keys: list) -> pd.Series:
    s = fs_col.loc[top_keys]
    other = max(1.0 - s.sum(), 0.0)
    n_other = len(U_COLS_FULL) - len(top_keys)
    return pd.concat(
        [s, pd.Series({f"other (remaining {n_other} parameters)": other})]
    )


stack_df = (
    pd.DataFrame(
        {
            "Year 1": stacked_series(fs_y1["pt_share_y1"], top_params),
            "Year 40": stacked_series(fs_y40["pt_share_y40"], top_params),
        }
    ).T
    * 100
)

stack_df = stack_df.rename(
    columns=lambda x: (
        f"{x} — {PARAMETER_LABELS[x]}" if x in PARAMETER_LABELS else x
    )
)

fig, ax = plt.subplots(figsize=(10, 5.5))
stack_df.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")

for container in ax.containers:
    labels = [f"{v:.1f}%" if v >= 2.0 else "" for v in container.datavalues]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8)

ax.set_ylabel("Feature score (%, normalised relative importance)")
ax.set_title(
    f"Extra-Trees Feature Scores for PT Share — Year 1 vs. Year 40\n"
    f"(Top {TOP_N} of {len(U_COLS_FULL)} candidate uncertainties)"
)
ax.tick_params(axis="x", rotation=0)
ax.legend(
    title="Parameter — Plain-English Meaning",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_feature_scores_stacked.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


## Morris screening → focused Sobol

A full Sobol analysis on all 16 parameters requires thousands of model evaluations. **Morris screening** is a structured, cheap method ($40 \times 17 = 680$ runs) used to rank parameters before running a full Sobol decomposition:

* **$\mu^*$ (mean absolute effect)**: Direct influence of each parameter on the outcome.
* **$\sigma$ (standard deviation)**: Non-linear effects and interactions between parameters.

We use Morris screening to select the top 6–8 candidate parameters for the focused Sobol pass.


In [ ]:
# 1. Run Morris design across all 16 candidate uncertainties
N_MORRIS = 40  # trajectories; total runs = 40 * (16 + 1) = 680

with SequentialEvaluator(ema_model_full) as evaluator:
  morris_experiments, morris_outcomes = perform_experiments(
      ema_model_full,
      N_MORRIS,
      evaluator=evaluator,
      uncertainty_sampling=Samplers.MORRIS,
      log_progress=True,
  )

print(
    f"Morris design: {N_MORRIS} trajectories x {len(U_COLS_FULL) + 1} ="
    f" {len(morris_experiments)} runs."
)

# 2. Compute SALib Morris sensitivity indices for Year 1 and Year 40
problem_full = get_SALib_problem(ema_model_full.uncertainties)
X_morris = morris_experiments[problem_full["names"]].values

morris_rows = []
for year in ("y1", "y40"):
  Si = morris_analyze.analyze(
      problem_full,
      X_morris,
      morris_outcomes[f"pt_share_{year}"],
      print_to_console=False,
  )
  for name, mu_star, sigma in zip(
      problem_full["names"], Si["mu_star"], Si["sigma"]
  ):
    morris_rows.append(
        {"year": year, "parameter": name, "mu_star": mu_star, "sigma": sigma}
    )

morris_df = pd.DataFrame(morris_rows)

morris_y40 = morris_df[morris_df["year"] == "y40"].sort_values(
    "mu_star", ascending=False
)
print("\nTop 10 parameters by Morris mu_star (Year 40, pt_share):")
print(
    morris_y40.head(10)[["parameter", "mu_star", "sigma"]]
    .round(4)
    .to_string(index=False)
)

N_REDUCED = 8
REDUCED_PARAMS = morris_y40.head(N_REDUCED)["parameter"].tolist()
print(f"\nReduced set for Sobol ({N_REDUCED} parameters):", REDUCED_PARAMS)


In [ ]:
# 3. Horizontal bar chart of mu* and sigma
top_morris = morris_y40.head(8).sort_values("mu_star")
labels = [
    f"{p} — {PARAMETER_LABELS.get(p, p)}" for p in top_morris["parameter"]
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(labels, top_morris["mu_star"], color="seagreen", alpha=0.85)

ax.errorbar(
    top_morris["mu_star"],
    labels,
    xerr=top_morris["sigma"],
    fmt="none",
    ecolor="dimgray",
    capsize=3,
    linewidth=1.2,
)

ax.set_xlabel("Morris μ* (Year-40 PT Share)")
ax.set_ylabel("")
ax.set_title("Morris Screening — Influential Uncertainties (Year 40)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_morris_screening.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


Now run a **focused Sobol** analysis restricted to the reduced set of `N_REDUCED` parameters. With 8 parameters instead of 16, running $N \times (2 \times 8 + 2) = 18N$ evaluations is computationally efficient at $N = 256$, providing converged first-order ($S_1$) and total-order ($S_T$) indices.


In [ ]:
# 1. Register reduced model
ema_model_reduced = Model("MehrSpurReduced", function=mehrspur_snapshot_model)
ema_model_reduced.uncertainties = [
    RealParameter(name, 0.0, 1.0) for name in REDUCED_PARAMS
]
ema_model_reduced.outcomes = [ScalarOutcome(o) for o in OUTCOMES]

N_SOBOL_REDUCED = 256
with SequentialEvaluator(ema_model_reduced) as evaluator:
  sob_experiments, sob_outcomes = perform_experiments(
      ema_model_reduced,
      N_SOBOL_REDUCED,
      evaluator=evaluator,
      uncertainty_sampling=Samplers.SOBOL,
      log_progress=True,
  )

print(
    f"Focused Sobol design: N={N_SOBOL_REDUCED} x {2 * N_REDUCED + 2} ="
    f" {len(sob_experiments)} runs "
    f"(vs. {N_SOBOL_REDUCED * (2 * len(U_COLS_FULL) + 2):,} for the full"
    f" 16-parameter space)."
)

# 2. Compute Sobol sensitivity indices for Year 1 and Year 40
problem_reduced = get_SALib_problem(ema_model_reduced.uncertainties)
sobol_rows = []
for year in ("y1", "y40"):
  Si = sobol_analyze.analyze(
      problem_reduced,
      sob_outcomes[f"pt_share_{year}"],
      calc_second_order=True,
      print_to_console=False,
  )
  for name, s1, st in zip(problem_reduced["names"], Si["S1"], Si["ST"]):
    sobol_rows.append({"year": year, "parameter": name, "S1": s1, "ST": st})

sobol_reduced_df = pd.DataFrame(sobol_rows)

sobol_pivot = sobol_reduced_df.pivot(
    index="parameter", columns="year", values="ST"
).sort_values("y40", ascending=False)
print("\nSobol total-order index (ST) for pt_share by snapshot year:")
print(sobol_pivot.round(5).to_string())


In [ ]:
from matplotlib.ticker import PercentFormatter

sobol_plot = sobol_reduced_df.pivot(
    index="parameter", columns="year", values="ST"
)
sobol_plot = sobol_plot.loc[sobol_plot.max(axis=1).sort_values().index]
sobol_plot.index = [
    f"{p} — {PARAMETER_LABELS.get(p, p)}" for p in sobol_plot.index
]

fig, ax = plt.subplots(figsize=(10, 5.5))
sobol_plot.plot(
    kind="barh", ax=ax, color={"y1": "steelblue", "y40": "darkorange"}
)

ax.set_xlabel("Sobol total-order index (ST)")
ax.set_ylabel("")
ax.set_title(
    "Sobol Sensitivity Indices for PT Share — Reduced Parameter Set\n(Year 1"
    " vs. Year 40)"
)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(["Year 1", "Year 40"], title="")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_sobol_reduced.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


In [ ]:
D = len(problem_reduced["names"])
block = 2 * D + 2
N_values = [
    n for n in [2, 4, 8, 16, 32, 64, 128, 256, 512] if n <= N_SOBOL_REDUCED
]
rows = []

for N in N_values:
  Si = sobol_analyze.analyze(
      problem_reduced,
      sob_outcomes["pt_share_y40"][: N * block],
      calc_second_order=True,
      print_to_console=False,
  )
  rows += [
      {"N": N, "parameter": p, "ST": st}
      for p, st in zip(problem_reduced["names"], Si["ST"])
  ]

sobol_convergence = pd.DataFrame(rows)
top = sobol_convergence.query("N == @N_values[-1]").nlargest(3, "ST")[
    "parameter"
]

fig, ax = plt.subplots(figsize=(9, 4.5))

for parameter in top:
  d = sobol_convergence.query("parameter == @parameter")
  label = f"{parameter} — {PARAMETER_LABELS.get(parameter, parameter)}"
  ax.plot(d["N"], d["ST"], marker="o", label=label)

ax.set_xlabel("Sobol base sample size N")
ax.set_ylabel("Total-order index (ST)")
ax.set_title("Convergence of Focused Sobol Sensitivity Indices (PT Share)")
ax.legend(
    title="Parameter — Plain-English Meaning",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
)
ax.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_sobol_convergence.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


### Compare Rankings: Extra-Trees vs. Sobol

We compare the top ranking uncertainties from **Extra-Trees feature scoring** against the **Sobol total-order ($S_T$) indices** to check for consistency across both sensitivity methods.


In [ ]:
# 1. Extract top-3 parameters from both methods for Year-40 PT Share
et = fs_y40["pt_share_y40"].nlargest(3)
sob = (
    sobol_reduced_df.query("year == 'y40'")
    .set_index("parameter")["ST"]
    .nlargest(3)
)

params = list(dict.fromkeys([*et.index, *sob.index]))
comparison = pd.DataFrame({
    "Extra-Trees": et.reindex(params, fill_value=0),
    "Sobol ST": sob.reindex(params, fill_value=0),
})

# Normalize separately so both methods share a comparable [0, 1] relative scale
comparison = comparison.div(comparison.max())
comparison.index = [
    f"{p} — {PARAMETER_LABELS.get(p, p)}" for p in comparison.index
]

fig, ax = plt.subplots(figsize=(8.5, 4.5))
comparison.plot(
    kind="barh", ax=ax, color=["steelblue", "darkorange"], width=0.7
)

plt.xlabel(
    "Relative importance (max score = 1.0)", fontsize=10, fontweight="bold"
)
plt.ylabel("")
plt.title(
    "Top Parameter Ranking Comparison — Year-40 PT Share\n(Extra-Trees"
    " Feature Scores vs. Sobol ST Indices)",
    fontsize=11,
    fontweight="bold",
)
plt.gca().invert_yaxis()
plt.legend(["Extra-Trees", "Sobol ST"], title="", loc="lower right")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_extratrees_vs_sobol.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


# Part 6 — Compare static infrastructure pathways under uncertainty
We evaluate the three **static pathways** defined in our project:
1. `baseline` (Stage 0: no new infrastructure for 40 years)
2. `static1` (Stage 1: Local Stations & Access Package from Year 1)
3. `static2` (Stage 2: Core Tunnel & Full Service Package from Year 1)

**Methodological requirement:** All three pathways must be evaluated on the **exact same uncertainty draws** across all scenarios ($400 \text{ scenarios} \times 3 \text{ pathways} = 1,200 \text{ runs}$) to ensure observed differences reflect infrastructure interventions rather than sampling noise.


In [ ]:
# 1. Register model with pathway categorical lever
ema_model_pathway = Model("MehrSpurPathway", function=mehrspur_snapshot_model)
ema_model_pathway.uncertainties = m.get_ema_uncertainties(include_nuisance=True)
ema_model_pathway.levers = [CategoricalParameter("pathway", STATIC_PATHWAYS)]
ema_model_pathway.outcomes = [ScalarOutcome(o) for o in OUTCOMES]

# 2. Define EMA policies for each static pathway
ema_policies = [Policy(key, **{"pathway": key}) for key in STATIC_PATHWAYS]

with SequentialEvaluator(ema_model_pathway) as evaluator:
  pathway_experiments, pathway_outcomes = perform_experiments(
      ema_model_pathway,
      N_SCENARIOS,
      policies=ema_policies,
      evaluator=evaluator,
      uncertainty_sampling=Samplers.LHS,
      log_progress=True,
  )

df_pathway = pathway_experiments.copy()
for col, arr in pathway_outcomes.items():
  df_pathway[col] = arr

print(
    f"Total runs: {len(df_pathway)}  ({N_SCENARIOS} scenarios x"
    f" {len(STATIC_PATHWAYS)} pathways)"
)

# 3. Verify identical uncertainty draws across pathways per scenario
check = df_pathway.pivot_table(
    index="scenario", columns="pathway", values="u_demand", observed=True
)
same_draws = check.nunique(axis=1).eq(1).all()
assert same_draws, "Pathways were not run on identical uncertainty draws."
print(
    "Checks passed: all three pathways share identical uncertainty draws per"
    " scenario."
)


## Outcome distributions by pathway

Boxplots of core transport outcomes (`pt_share`, `avg_travel_time_min`, `congestion_delay_hours`), split by static pathway across both snapshot years (Year 1 vs. Year 40).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
PATHWAY_COLORS = {
    "baseline": "steelblue",
    "static1": "seagreen",
    "static2": "darkorange",
}
OBJECTIVES = ["pt_share", "avg_travel_time_min", "congestion_delay_hours"]
OBJ_LABELS = {
    "pt_share": "PT Share (PKM >5km)",
    "avg_travel_time_min": "Avg Travel Time (min)",
    "congestion_delay_hours": "Congestion Delay (hours)",
}


for row_idx, year in enumerate(["y1", "y40"]):
  year_label = "Year 1" if year == "y1" else "Year 40"
  for col_idx, obj in enumerate(OBJECTIVES):
    ax = axes[row_idx, col_idx]
    data = [
        df_pathway.loc[df_pathway["pathway"] == key, f"{obj}_{year}"]
        for key in STATIC_PATHWAYS
    ]
    bp = ax.boxplot(data, tick_labels=STATIC_PATHWAYS, patch_artist=True)
    for patch, key in zip(bp["boxes"], STATIC_PATHWAYS):
      patch.set_facecolor(PATHWAY_COLORS[key])
      patch.set_alpha(0.75)
    ax.set_title(f"{OBJ_LABELS[obj]} — {year_label}", fontsize=11)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Outcome Distributions by Pathway Across Sampled Futures",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_pathway_distributions.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


Alternative grouping by pathway.

In [ ]:
def p10(s):
  return s.quantile(0.10)

def p90(s):
  return s.quantile(0.90)

range_table = (
    df_pathway.groupby("pathway", observed=True)[[
        "car_share_y40",
        "pt_share_y40",
        "avg_travel_time_min_y40",
        "congestion_delay_hours_y40",
        "co2_tonnes_y40",
        "total_demand_y40",
    ]]
    .agg(["median", p10, p90])
    .reindex(STATIC_PATHWAYS)
)

display(range_table.round(3))


In [ ]:
PC_COLS = [
    f"{obj}_{year}"
    for obj in ["pt_share", "avg_travel_time_min", "congestion_delay_hours"]
    for year in ["y1", "y40"]
]

PC_LABELS = [
    "PT share (PKM)\nYear 1",
    "PT share (PKM)\nYear 40",
    "Avg travel time\nYear 1",
    "Avg travel time\nYear 40",
    "Congestion delay\nYear 1",
    "Congestion delay\nYear 40",
]

pc_df = df_pathway[["pathway"] + PC_COLS].copy()
for col in PC_COLS:
  lo, hi = pc_df[col].min(), pc_df[col].max()
  pc_df[col] = (pc_df[col] - lo) / (hi - lo) if hi > lo else 0.5

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(PC_COLS))

for pathway in STATIC_PATHWAYS:
  data = pc_df.loc[pc_df["pathway"] == pathway, PC_COLS]
  ax.plot(x, data.T, color=PATHWAY_COLORS[pathway], alpha=0.08, linewidth=0.8)
  ax.plot(
      x,
      data.median(),
      color=PATHWAY_COLORS[pathway],
      linewidth=3,
      marker="o",
      label=f"{pathway} median",
  )

ax.set_xticks(x, PC_LABELS)
ax.set_ylabel("Normalized outcome (0 = pooled min, 1 = pooled max)")
ax.set_title(
    "Parallel Coordinates — Pathway Performance Across Sampled Futures"
)
ax.legend(title="Pathway", loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_pathway_parallel_coordinates.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


Each thin line represents one sampled future under one pathway. The thick lines show the median pathway profiles. Outcomes are normalized using the minimum and maximum across all pathways so that indicators with different units can be displayed on the same axes. Greater vertical separation indicates larger differences between pathways, while extensive overlap indicates that uncertainty is large relative to the pathway effect.


In [ ]:
PC_COLS = [
    f"{obj}_{year}"
    for obj in ["pt_share", "avg_travel_time_min", "congestion_delay_hours"]
    for year in ["y1", "y40"]
]
PC_LABELS = [
    "PT share (PKM)\nYear 1",
    "PT share (PKM)\nYear 40",
    "Avg travel time\nYear 1",
    "Avg travel time\nYear 40",
    "Congestion delay\nYear 1",
    "Congestion delay\nYear 40",
]

pc_df = df_pathway[["pathway"] + PC_COLS].copy()
for col in PC_COLS:
  lo, hi = pc_df[col].min(), pc_df[col].max()
  pc_df[col] = (pc_df[col] - lo) / (hi - lo) if hi > lo else 0.5

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(PC_COLS))
offsets = dict(zip(STATIC_PATHWAYS, [-0.08, 0, 0.08]))

for pathway in STATIC_PATHWAYS:
  data = pc_df.loc[pc_df["pathway"] == pathway, PC_COLS]
  p10, p25, p50, p75, p90 = data.quantile([0.10, 0.25, 0.50, 0.75, 0.90]).values
  color, xs = PATHWAY_COLORS[pathway], x + offsets[pathway]

  ax.plot(xs, p50, color=color, linewidth=2.5, alpha=0.8, label=pathway)
  ax.vlines(xs, p10, p90, color=color, linewidth=1.2)
  ax.vlines(xs, p25, p75, color=color, linewidth=5)

  for xi, q10, q25, q50, q75, q90 in zip(xs, p10, p25, p50, p75, p90):
    ax.hlines([q10, q90], xi - 0.025, xi + 0.025, color=color, linewidth=1.2)
    ax.hlines([q25, q75], xi - 0.035, xi + 0.035, color=color, linewidth=2)
    ax.hlines(q50, xi - 0.045, xi + 0.045, color="white", linewidth=2.5)
    ax.hlines(q50, xi - 0.045, xi + 0.045, color=color, linewidth=1)

ax.set_xticks(x, PC_LABELS)
ax.set_ylabel("Normalized outcome (0 = pooled min, 1 = pooled max)")
ax.set_title("Parallel Coordinates — Pathway Medians and Percentile Ranges")
ax.legend(title="Pathway", loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_pathway_parallel_coordinates.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


## Feature scores by pathway

Does *which* uncertainty matters most for public transport share change once major infrastructure (`static1`, `static2`) is constructed? We repeat the Extra-Trees feature scoring separately for each pathway.


In [ ]:
U_COLS_PATHWAY = [u.name for u in ema_model_pathway.uncertainties]
scores = {}

for pathway in STATIC_PATHWAYS:
  sub = df_pathway[df_pathway["pathway"] == pathway]
  fs = feature_scoring.get_feature_scores_all(
      sub[U_COLS_PATHWAY], {"pt_share_y40": sub["pt_share_y40"].values}
  )
  scores[pathway] = fs["pt_share_y40"]

score_df = pd.DataFrame(scores) * 100
top8 = score_df.max(axis=1).nlargest(8).sort_values().index
plot_df = score_df.loc[top8].copy()
plot_df.index = [f"{p} — {PARAMETER_LABELS.get(p, p)}" for p in top8]

fig, ax = plt.subplots(figsize=(11, 6))
plot_df.plot(
    kind="barh",
    ax=ax,
    width=0.8,
    color=[PATHWAY_COLORS[key] for key in STATIC_PATHWAYS],
)

for container in ax.containers:
  labels = [f"{v:.1f}%" if v >= 0.5 else "" for v in container.datavalues]
  ax.bar_label(container, labels=labels, padding=3, fontsize=8)

ax.set_xlabel("Feature score (%)", fontsize=10, fontweight="bold")
ax.set_ylabel("")
ax.set_title(
    "Key Drivers of Year-40 PT Share by Pathway\n(Extra-Trees Feature Scoring)",
    fontsize=11,
    fontweight="bold",
)
ax.legend(title="Pathway", loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_feature_scores_by_pathway.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


# Part 7 — Scenario discovery: understanding vulnerability

*"Under which combinations of uncertain inputs does the system fail to meet a target?"*

We now ask: are there specific, describable regions in the uncertainty space where the system fails to achieve its planning goals? We apply the **Patient Rule Induction Method (PRIM)** to discover failure boundaries in Year 40.

## Defining the vulnerability
The regional transport objective sets a target of **35% public transport mode share by Year 40** (as specified by `PT_SHARE_TARGET` in `parameters.py`). We define vulnerability as failing to reach this minimum transit share:
$$\text{vulnerable} \iff \text{pt\_share}_{\text{Year 40}} < 0.35$$

> 💡 **Why 35%?** Because our primary modal split is **PKM-weighted (>5km)**, this 35% threshold represents a realistic, strategic benchmark for intercity travel where the railway successfully captures passenger-distance from the A1 highway.



In [ ]:
# 1. Flag vulnerable scenarios dynamically based on threshold
VULNERABILITY_THRESHOLD = getattr(p, "PT_SHARE_TARGET", 0.35)
target_pct = int(round(VULNERABILITY_THRESHOLD * 100))
df_pathway["vulnerable"] = df_pathway["pt_share_y40"] < VULNERABILITY_THRESHOLD

overall_rate = df_pathway["vulnerable"].mean()
print(
    f"Overall failure rate across all {len(df_pathway)} pathway-scenario runs: {overall_rate:.0%}"
)

fail_by_pathway = (
    df_pathway.groupby("pathway", observed=True)["vulnerable"]
    .mean()
    .reindex(STATIC_PATHWAYS)
)
print(
    f"\nFailure rate by pathway (share of sampled futures below {target_pct}% PT share at Year 40):"
)
print((fail_by_pathway * 100).round(1).astype(str) + "%")

# 2. Visualize vulnerability distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left Plot: Pie Chart (handles 0% and 100% cleanly) ---
pie_vals = [overall_rate, 1 - overall_rate]
pie_lbls = [f"Below {target_pct}% target", "Meets target"]
pie_cols = ["crimson", "lightgray"]

# Filter out 0 slices to prevent overlapping text
active_indices = [i for i, v in enumerate(pie_vals) if v > 1e-6]
plot_vals = [pie_vals[i] for i in active_indices]
plot_lbls = [pie_lbls[i] for i in active_indices]
plot_cols = [pie_cols[i] for i in active_indices]

axes[0].pie(
    plot_vals,
    labels=plot_lbls,
    autopct="%1.0f%%",
    colors=plot_cols,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[0].set_title(f"All pathways pooled (n={len(df_pathway)})", fontsize=11)

# --- Right Plot: Bar Chart ---
bars = axes[1].bar(
    STATIC_PATHWAYS,
    fail_by_pathway.values * 100,
    color=[PATHWAY_COLORS[key] for key in STATIC_PATHWAYS],
    width=0.55,
)

# Text labels on top of bars
for bar, val in zip(bars, fail_by_pathway.values):
    height = bar.get_height()
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        height + 2.0,
        f"{val:.0%}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

# Fix Y-axis to standard 0-100% range
axes[1].set_ylim(0, 105)
axes[1].set_ylabel(f"Share of scenarios below {target_pct}% PT share at Year 40 (%)")
axes[1].set_title("Failure rate by pathway", fontsize=11)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle(
    f"How often, and for whom, does the corridor miss the {target_pct}% PT share target?",
    fontsize=12,
    fontweight="bold",
    y=1.03,
)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_vulnerability_by_pathway.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


## Why do these failures happen? PRIM on the baseline pathway

**PRIM (Patient Rule Induction Method)** searches for a "box" — simple rectangular bounds on the uncertain inputs — that captures a high concentration (**density**) of vulnerable scenarios while covering a large proportion of all failures (**coverage**).

We restrict the search to the two structural macro-uncertainties, **`u_beta_pt`** and **`u_demand`**, which govern deep future societal trends. We run PRIM specifically on **`baseline`** (Stage 0) to understand under which future conditions doing nothing fails.


In [ ]:
MIN_FAILURES_FOR_PRIM = 15

base_df = df_pathway[df_pathway["pathway"] == "baseline"]
x_prim = base_df[["u_beta_pt", "u_demand"]]
y_prim = base_df["vulnerable"].values
n_failures = int(y_prim.sum())

print(
    f"Baseline: {n_failures} failing scenarios out of {len(y_prim)} "
    f"({n_failures / len(y_prim):.0%}) -- enough to run PRIM."
    if n_failures >= MIN_FAILURES_FOR_PRIM
    else f"Baseline: only {n_failures} failing scenarios -- too few for a"
    " reliable PRIM box."
)

# 1. Run PRIM box search (peel_alpha=0.1)
prim_alg = prim.Prim(x_prim, y_prim, peel_alpha=0.1)
box_baseline = prim_alg.find_box()

# 2. Display peeling tradeoff curve (density vs. coverage)
box_baseline.show_tradeoff()
plt.title(
    "PRIM Peeling Trajectory — Baseline Pathway", fontsize=11, fontweight="bold"
)
plt.tight_layout()

plt.savefig(
    PROJECT_ROOT / "figures" / "03_prim_tradeoff.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

# 3. Print discovered box bounds
box_baseline.inspect(style="table")
bounds_baseline = prim_alg.boxes_to_dataframe()
print("\nDiscovered Box Bounds:")
print(bounds_baseline)


### Interpreting the PRIM peeling trajectory

PRIM begins with a box containing all sampled scenarios and progressively removes ("peels") small slices from the edges of the uncertainty space. At each step, it selects the peel that produces the highest concentration of vulnerable scenarios inside the remaining box.

The peeling trajectory shows the trade-off between:
- **Coverage:** the share of all vulnerable scenarios captured by the box.
- **Density:** the share of scenarios inside the box that are vulnerable.
- **Mass:** the share of all sampled scenarios contained in the box.

The selected box balances a high failure concentration (density) with capturing a meaningful share of all failures (coverage).

### A clearer picture of the box

The chart below visualizes the discovered failure region: every sampled scenario plotted by `(u_demand, u_beta_pt)`, coloured by pass/fail, with the PRIM box overlaid as a dashed rectangle.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

fig = plt.figure(figsize=(14, 8.5))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 3], hspace=0.32, wspace=0.10)

ax_beta = fig.add_subplot(gs[0, 0])
ax_demand = fig.add_subplot(gs[0, 1])
axes = [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]

palette = {False: "lightgray", True: "crimson"}

# 1. Top: marginal uncertainty distributions by vulnerability status
sns.kdeplot(
    data=base_df,
    x="u_beta_pt",
    hue="vulnerable",
    palette=palette,
    fill=True,
    alpha=0.3,
    common_norm=False,
    cut=0,
    legend=False,
    ax=ax_beta,
)
sns.kdeplot(
    data=base_df,
    x="u_demand",
    hue="vulnerable",
    palette=palette,
    fill=True,
    alpha=0.3,
    common_norm=False,
    cut=0,
    legend=False,
    ax=ax_demand,
)

ax_beta.set_title(
    "Distribution by PT Preference Shift (u_beta_pt)", fontsize=10
)
ax_demand.set_title(
    "Distribution by Corridor Demand Growth (u_demand)", fontsize=10
)
ax_beta.set_xlabel("u_beta_pt (0 = weak PT affinity, 1 = strong PT affinity)")
ax_demand.set_xlabel("u_demand (0 = low growth, 1 = high growth)")

for ax in [ax_beta, ax_demand]:
  ax.set_xlim(0, 1)
  ax.set_ylabel("Density")
  ax.grid(alpha=0.3)

# 2. Bottom-left: individual sampled scenarios
ax = axes[0]
ax.scatter(
    base_df.loc[~base_df["vulnerable"], "u_demand"],
    base_df.loc[~base_df["vulnerable"], "u_beta_pt"],
    color="lightgray",
    s=22,
    alpha=0.8,
)
ax.scatter(
    base_df.loc[base_df["vulnerable"], "u_demand"],
    base_df.loc[base_df["vulnerable"], "u_beta_pt"],
    color="crimson",
    s=22,
    alpha=0.8,
)
ax.set_title("Individual sampled futures", fontsize=11)

# 3. Bottom-right: vulnerability rate within each 5x5 grid cell
bins = np.linspace(0, 1, 6)
total, xedges, yedges = np.histogram2d(
    base_df["u_demand"], base_df["u_beta_pt"], bins=[bins, bins]
)
failed, _, _ = np.histogram2d(
    base_df.loc[base_df["vulnerable"], "u_demand"],
    base_df.loc[base_df["vulnerable"], "u_beta_pt"],
    bins=[bins, bins],
)
density = np.divide(
    failed, total, out=np.full_like(failed, np.nan), where=total > 0
)

ax = axes[1]
mesh = ax.pcolormesh(
    xedges, yedges, density.T, cmap="Reds", vmin=0, vmax=1, shading="flat"
)
cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label(f"Share below {target_pct}% PT target")
ax.set_title("Local vulnerability rate (5 × 5 grid)", fontsize=11)

# 4. Overlaid PRIM box on bottom panels
box_limits = box_baseline.box_lims[-1]
u_beta_lo, u_beta_hi = box_limits["u_beta_pt"]
u_demand_lo, u_demand_hi = box_limits["u_demand"]

box_args = (
    (u_demand_lo, u_beta_lo),
    u_demand_hi - u_demand_lo,
    u_beta_hi - u_beta_lo,
)

for ax in axes:
  ax.add_patch(
      Rectangle(
          *box_args,
          fill=False,
          edgecolor="black",
          linewidth=2.5,
          linestyle="--",
          clip_on=False,
      )
  )
  ax.set_xlabel("u_demand (0 = low growth, 1 = high growth)")
  ax.set_xlim(-0.03, 1.03)
  ax.set_ylim(-0.03, 1.03)
  ax.grid(alpha=0.3)

axes[0].set_ylabel(
    "u_beta_pt (0 = weak PT affinity, 1 = strong PT affinity)"
)
axes[1].tick_params(axis="y", labelleft=False)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="none", color="lightgray", markersize=7, label=f"Meets {target_pct}% target"),
    Line2D([0], [0], marker="o", linestyle="none", color="crimson", markersize=7, label=f"Below {target_pct}% target"),
    Rectangle((0, 0), 1, 1, fill=False, edgecolor="black", linewidth=2, linestyle="--", label="PRIM box"),
]

fig.suptitle(
    f"Baseline Pathway — Where the {target_pct}% PT Target is Missed\n"
    f"PRIM Box: coverage={box_baseline.coverage:.0%}, density={box_baseline.density:.0%}",
    fontsize=12,
    fontweight="bold",
    y=0.99,
)

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.91),
    ncol=3,
    frameon=False,
    fontsize=9,
)
fig.subplots_adjust(top=0.82, bottom=0.08, left=0.08, right=0.93)

plt.savefig(
    PROJECT_ROOT / "figures" / "03_prim_box_overlay.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


### Do `static1` and `static2` still have a vulnerability worth explaining?

We repeat the PRIM search for the remaining static pathways to test if expanding infrastructure completely eliminates the vulnerability region or merely shrinks it.


In [ ]:
for pathway in ["static1", "static2"]:
  sub = df_pathway[df_pathway["pathway"] == pathway]
  n_fail = int(sub["vulnerable"].sum())
  print(
      f"{pathway}: {n_fail} failing scenarios out of {len(sub)}"
      f" ({n_fail / len(sub):.0%})."
  )

  if n_fail < MIN_FAILURES_FOR_PRIM:
    print(
        "  -> Too few failures for a meaningful PRIM box. That scarcity IS the"
        " key finding:"
    )
    print(
        f"     Once {pathway} is constructed, missing the 35% PT share target is"
        " a rare edge case,"
    )
    print("     rather than a broad vulnerability in the uncertainty space.\n")
  else:
    x_s = sub[["u_beta_pt", "u_demand"]]
    y_s = sub["vulnerable"].values
    prim_s = prim.Prim(x_s, y_s, peel_alpha=0.1)
    box_s = prim_s.find_box()
    print(
        f"  -> PRIM box found: coverage={box_s.coverage:.0%},"
        f" density={box_s.density:.0%}\n"
    )
    print(prim_s.boxes_to_dataframe())
    print()


## Export results

Save the pathway-comparison dataset and vulnerability/PRIM summary for reuse. The exported schema uses `pathway` consistently with `code/pathways.py`.


In [ ]:
import json

df_pathway.to_csv(RESULTS_DIR / "03_experiments_outcomes.csv", index=False)

box_limits = box_baseline.box_lims[-1]
u_beta_lims = [float(box_limits["u_beta_pt"][0]), float(box_limits["u_beta_pt"][1])]
u_demand_lims = [float(box_limits["u_demand"][0]), float(box_limits["u_demand"][1])]
vulnerability_summary = {
    "n_scenarios": N_SCENARIOS,
    "pathways": STATIC_PATHWAYS,
    "outcomes": OUTCOMES,
    "n_uncertainties": len(U_COLS_FULL),
    "vulnerability": {
        "definition": f"pt_share_y40 < {VULNERABILITY_THRESHOLD:.2f} (Year 40 PT Share Target)",
        "threshold": VULNERABILITY_THRESHOLD,
        "overall_failure_rate": float(overall_rate),
        "failure_rate_by_pathway": {
            key: float(fail_by_pathway[key]) for key in STATIC_PATHWAYS
        },
    },
    "prim_box_baseline": {
        "coverage": float(box_baseline.coverage),
        "density": float(box_baseline.density),
        "u_beta_range": u_beta_lims,
        "u_demand_range": u_demand_lims,
    },
}

with open(RESULTS_DIR / "03_vulnerability_definitions.json", "w", encoding="utf-8") as f:
    json.dump(vulnerability_summary, f, indent=2)
print("Exported to", RESULTS_DIR)

In [ ]:
#total runtime notebook03
print(f"__NOTEBOOK_RUNTIME_SECONDS__={_time.perf_counter() - _NB_START:.3f}")
print(f"__NOTEBOOK_RUNTIME_MINUTES__={(_time.perf_counter() - _NB_START) / 60:.3f}")
